# 🚀 Medical App Deployment
Run the cells to generate `app.py`, `streamlit_app.py` and `requirements.txt`.

Requires `medical_model.pth` and `eval_metrics.json` (from the training notebook) in the same folder.

Then run in two terminals:
```
uvicorn app:app --reload --port 8000
streamlit run streamlit_app.py
```

In [ ]:
%%writefile app.py
"""
app.py — FastAPI backend
Loads medical_model.pth, runs a QA check, DenseNet121 inference and Grad-CAM.
Run:  uvicorn app:app --reload --port 8000
"""

import io
import base64
from typing import Optional

import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torchvision import models, transforms
from fastapi import FastAPI, File, UploadFile, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel

# ── Config ───────────────────────────────────────────────────
MODEL_PATH = "medical_model.pth"
IMG_SIZE = 224
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CLASS_NAMES = ["NORMAL", "PNEUMONIA"]
LAP_THRESH = 20.0


# ── Model ────────────────────────────────────────────────────
def load_model():
    """Same DenseNet121 architecture as training, with saved weights."""
    m = models.densenet121(weights=None)
    m.classifier = nn.Sequential(
        nn.Linear(m.classifier.in_features, 256),
        nn.ReLU(),
        nn.Dropout(0.4),
        nn.Linear(256, 2),
    )
    m.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
    m.to(DEVICE)
    m.eval()
    return m


model = load_model()

tfm = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])


# ── QA filter ────────────────────────────────────────────────
def qa_ok(gray: np.ndarray):
    """Blur / exposure check. Returns (passed, reason)."""
    lap = float(cv2.Laplacian(gray, cv2.CV_64F).var())
    if lap < LAP_THRESH:
        return False, f"BLURRY (lap={lap:.1f})"
    mean_px = float(np.mean(gray))
    if mean_px < 10:
        return False, "TOO_DARK"
    if mean_px > 245:
        return False, "TOO_BRIGHT"
    return True, "OK"


# ── Grad-CAM ─────────────────────────────────────────────────
class GradCAM:
    def __init__(self, model, layer):
        self.model = model
        self.act = None
        self.grad = None
        layer.register_forward_hook(
            lambda m, i, o: setattr(self, "act", o.detach()))
        layer.register_full_backward_hook(
            lambda m, gi, go: setattr(self, "grad", go[0].detach()))

    def generate(self, tensor, class_idx):
        self.model.zero_grad()
        logits = self.model(tensor)
        logits[0, class_idx].backward()
        weights = self.grad[0].mean(dim=(1, 2))
        cam = torch.zeros(self.act.shape[2:], device=DEVICE)
        for i, w in enumerate(weights):
            cam += w * self.act[0, i]
        cam = F.relu(cam)
        cam = cam - cam.min()
        if cam.max() > 0:
            cam = cam / cam.max()
        return cv2.resize(cam.cpu().numpy(), (IMG_SIZE, IMG_SIZE))


gc = GradCAM(model, model.features.denseblock4)


def arr_to_b64(arr: np.ndarray) -> str:
    buf = io.BytesIO()
    Image.fromarray(arr.astype(np.uint8)).save(buf, format="PNG")
    return base64.b64encode(buf.getvalue()).decode()


# ── Schema ───────────────────────────────────────────────────
class Result(BaseModel):
    predicted_class: str
    confidence: float
    normal_prob: float
    pneumonia_prob: float
    qa_passed: bool
    qa_reason: str
    overlay_b64: Optional[str] = None


# ── App ──────────────────────────────────────────────────────
app = FastAPI(
    title="Chest X-Ray Classifier",
    description="DenseNet121 + QA filter + Grad-CAM",
    version="1.0.0",
)
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)


@app.get("/health")
def health():
    return {"status": "ok", "device": str(DEVICE)}


@app.post("/predict", response_model=Result)
async def predict(file: UploadFile = File(...)):
    raw = await file.read()
    arr = np.frombuffer(raw, np.uint8)
    bgr = cv2.imdecode(arr, cv2.IMREAD_COLOR)
    if bgr is None:
        raise HTTPException(400, "Cannot read image file.")

    # QA filter
    gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
    gray = cv2.resize(gray, (IMG_SIZE, IMG_SIZE))
    gray = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(gray)
    passed, reason = qa_ok(gray)
    if not passed:
        return Result(predicted_class="REJECTED", confidence=0,
                      normal_prob=0, pneumonia_prob=0,
                      qa_passed=False, qa_reason=reason)

    # Inference
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    pil = Image.fromarray(rgb)
    tensor = tfm(pil).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        probs = torch.softmax(model(tensor), dim=1)[0]
    idx = probs.argmax().item()

    # Grad-CAM overlay
    heatmap = gc.generate(tfm(pil).unsqueeze(0).to(DEVICE), idx)
    orig = cv2.resize(rgb, (IMG_SIZE, IMG_SIZE))
    hmap = cv2.applyColorMap(np.uint8(255 * heatmap), cv2.COLORMAP_JET)
    hmap = cv2.cvtColor(hmap, cv2.COLOR_BGR2RGB)
    overlay = cv2.addWeighted(orig, 0.55, hmap, 0.45, 0)

    return Result(
        predicted_class=CLASS_NAMES[idx],
        confidence=round(probs[idx].item(), 4),
        normal_prob=round(probs[0].item(), 4),
        pneumonia_prob=round(probs[1].item(), 4),
        qa_passed=True,
        qa_reason="OK",
        overlay_b64=arr_to_b64(overlay),
    )


In [ ]:
%%writefile streamlit_app.py
"""
streamlit_app.py — Clinician portal
Upload an X-Ray -> prediction + Grad-CAM overlay.
Run:  streamlit run streamlit_app.py   (API must be running on :8000)
"""

import io
import base64
import json

import requests
import streamlit as st
from PIL import Image

st.set_page_config(page_title="X-Ray Classifier", page_icon="🫁", layout="wide")

API_URL = "http://localhost:8000/predict"


def b64_to_image(b64: str) -> Image.Image:
    return Image.open(io.BytesIO(base64.b64decode(b64)))


def prob_bar(label, value, color):
    pct = int(value * 100)
    st.markdown(
        f"""
        <div style="margin-bottom:8px">
          <span style="color:#AAAAAA">{label}</span>
          <div style="background:#1A1D23;border-radius:6px;height:24px;margin-top:4px">
            <div style="width:{pct}%;background:{color};height:24px;border-radius:6px;
                        text-align:right;padding-right:8px;line-height:24px;
                        color:white;font-weight:bold;font-size:13px">{pct}%</div>
          </div>
        </div>
        """,
        unsafe_allow_html=True,
    )


# ── Sidebar ──────────────────────────────────────────────────
with st.sidebar:
    st.header("ℹ️ Model Info")
    st.markdown(
        "**Model**: DenseNet121  \n"
        "**Task**: NORMAL vs PNEUMONIA  \n"
        "**XAI**: Grad-CAM  \n"
        "**QA**: Laplacian blur filter"
    )
    st.divider()
    st.markdown("🔴 **Red / Yellow** = model focused here  \n"
                "🔵 **Blue** = model ignored here")
    st.divider()
    try:
        with open("eval_metrics.json") as f:
            m = json.load(f)
        st.header("📊 Test Set Performance")
        st.metric("Accuracy", f"{m['accuracy'] * 100:.1f}%")
        st.metric("Sensitivity", f"{m['sensitivity'] * 100:.1f}%")
        st.metric("Specificity", f"{m['specificity'] * 100:.1f}%")
        st.metric("ROC-AUC", f"{m['roc_auc']:.3f}")
    except FileNotFoundError:
        st.info("Run the training notebook first to generate eval_metrics.json.")
    st.caption("⚠️ Research only. Not a medical device.")

# ── Main ─────────────────────────────────────────────────────
st.title("🫁 Chest X-Ray Pneumonia Classifier")
st.markdown(
    "Upload a chest X-Ray to get an AI prediction with a **Grad-CAM heatmap** "
    "showing which regions influenced the decision."
)
st.divider()

uploaded = st.file_uploader("📂 Upload Chest X-Ray (.jpg / .png)",
                            type=["jpg", "jpeg", "png"])

if uploaded:
    raw_bytes = uploaded.read()
    orig_img = Image.open(io.BytesIO(raw_bytes)).convert("RGB")

    with st.spinner("🔍 Analysing image ..."):
        try:
            res = requests.post(
                API_URL,
                files={"file": ("xray.jpg", raw_bytes, "image/jpeg")},
                timeout=60,
            ).json()
        except requests.exceptions.ConnectionError:
            st.error("❌ API not running. Start it with:\n"
                     "```\nuvicorn app:app --reload --port 8000\n```")
            st.stop()

    if not res["qa_passed"]:
        st.error(f"🚫 Image rejected by QA filter: `{res['qa_reason']}`")
        st.info("Please upload a clear, well-exposed chest X-Ray.")
        st.stop()

    st.success("✅ Image passed QA validation")

    pred = res["predicted_class"]
    bg = "#1A4731" if pred == "NORMAL" else "#4A1A1A"
    tc = "#2ECC71" if pred == "NORMAL" else "#E74C3C"

    st.subheader("🩺 Result")
    st.markdown(
        f"""
        <div style="background:{bg};border-radius:12px;padding:24px;
                    text-align:center;margin-bottom:20px">
          <div style="font-size:34px;font-weight:bold;color:{tc}">{pred}</div>
          <div style="color:#AAAAAA;margin-top:6px">
            Confidence: {res['confidence'] * 100:.1f}%
          </div>
        </div>
        """,
        unsafe_allow_html=True,
    )
    prob_bar("NORMAL", res["normal_prob"], "#2ECC71")
    prob_bar("PNEUMONIA", res["pneumonia_prob"], "#E74C3C")

    st.divider()
    st.subheader("🔬 Grad-CAM Overlay")
    st.caption("Red/yellow = high attention | Blue = low attention")

    col1, col2 = st.columns(2)
    with col1:
        st.markdown("**Original X-Ray**")
        st.image(orig_img, use_container_width=True)
    with col2:
        st.markdown("**Grad-CAM Overlay**")
        st.image(b64_to_image(res["overlay_b64"]), use_container_width=True)

    with st.expander("🛠️ Raw API response"):
        st.json({k: v for k, v in res.items() if k != "overlay_b64"})


In [ ]:
%%writefile requirements.txt
torch>=2.2.0
torchvision>=0.17.0
opencv-python-headless>=4.9.0
Pillow>=10.2.0
numpy>=1.26.4,<2
scikit-learn>=1.4.1
matplotlib>=3.8.3
seaborn>=0.13.2
tqdm>=4.66.2
fastapi>=0.110.0
uvicorn>=0.29.0
python-multipart>=0.0.9
pydantic>=2.6.4
streamlit>=1.33.0
requests>=2.31.0


In [ ]:
from pathlib import Path
for f in ["app.py","streamlit_app.py","requirements.txt","medical_model.pth","eval_metrics.json"]:
    print("✅" if Path(f).exists() else "❌", f)
